In [6]:
import pandas as pd

df = pd.read_csv("../data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [7]:
# Drop constant columns (no variance in this dataset)
df.drop(columns=['EmployeeCount', 'StandardHours', 'Over18'], inplace=True)

# Drop duplicates
df.drop_duplicates(inplace=True)

# Encode target variable
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# Encode binary categorical column
df['OverTime'] = df['OverTime'].map({'Yes': 1, 'No': 0})

# One-hot encode remaining categoricals
df = pd.get_dummies(df, columns=['BusinessTravel', 'Department', 'EducationField',
                                  'Gender', 'JobRole', 'MaritalStatus'],
                    drop_first=True)

print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()


Shape: (1470, 46)
Missing values: 0


,Age,Attrition,DailyRate,DistanceFromHome,Education,EmployeeNumber,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,...,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single
0,41,1,1102,1,2,1,2,94,3,2,...,False,False,False,False,False,False,True,False,False,True
1,49,0,279,8,1,2,3,61,2,2,...,False,False,False,False,False,True,False,False,True,False
2,37,1,1373,2,2,4,4,92,2,1,...,False,True,False,False,False,False,False,False,False,True
3,33,0,1392,3,4,5,4,56,3,1,...,False,False,False,False,False,True,False,False,True,False
4,27,0,591,2,1,7,1,40,3,1,...,False,True,False,False,False,False,False,False,True,False


In [8]:
# Ratio features
df['PromotionToTenureRatio']  = df['YearsSinceLastPromotion'] / (df['YearsAtCompany'] + 1)
df['IncomePerYearAtCompany']  = df['MonthlyIncome'] / (df['YearsAtCompany'] + 1)

# Tenure bucket → OHE immediately
df['TenureBucket'] = pd.cut(
    df['YearsAtCompany'],
    bins=[-1, 2, 5, 10, float('inf')],
    labels=['0-2', '3-5', '6-10', '10+']
)
df = pd.get_dummies(df, columns=['TenureBucket'], drop_first=True)
tenure_dummies = [c for c in df.columns if c.startswith('TenureBucket_')]
for col in tenure_dummies:
    df[col] = df[col].astype(int)

# Overtime flag (explicit alias of the already-binary OverTime column)
df['OvertimeFlag'] = df['OverTime']

# Interaction features
df['OverTime_x_JobSatisfaction']          = df['OverTime'] * df['JobSatisfaction']
df['DistanceFromHome_x_WorkLifeBalance']  = df['DistanceFromHome'] * df['WorkLifeBalance']

new_cols = ['PromotionToTenureRatio', 'IncomePerYearAtCompany',
            'OvertimeFlag', 'OverTime_x_JobSatisfaction',
            'DistanceFromHome_x_WorkLifeBalance'] + tenure_dummies
print(f"Shape after feature engineering: {df.shape}")
df[new_cols].head()


Shape after feature engineering: (1470, 54)


,PromotionToTenureRatio,IncomePerYearAtCompany,OvertimeFlag,OverTime_x_JobSatisfaction,DistanceFromHome_x_WorkLifeBalance,TenureBucket_3-5,TenureBucket_6-10,TenureBucket_10+
0,0.000000,856.142857,1,4,1,0,1,0
1,0.090909,466.363636,0,0,24,0,1,0
2,0.000000,2090.000000,1,3,6,0,0,0
3,0.333333,323.222222,1,3,9,0,1,0
4,0.666667,1156.000000,0,0,6,0,0,0


In [9]:
ordinal_maps = {
    'Education':              {1: 'Below College', 2: 'College', 3: 'Bachelor', 4: 'Master', 5: 'Doctor'},
    'EnvironmentSatisfaction':{1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'JobInvolvement':         {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'JobSatisfaction':        {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'PerformanceRating':      {1: 'Low', 2: 'Good', 3: 'Excellent', 4: 'Outstanding'},
    'RelationshipSatisfaction':{1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'WorkLifeBalance':        {1: 'Bad', 2: 'Good', 3: 'Better', 4: 'Best'},
}

ordinal_orders = {
    'Education':              ['Below College', 'College', 'Bachelor', 'Master', 'Doctor'],
    'EnvironmentSatisfaction':['Low', 'Medium', 'High', 'Very High'],
    'JobInvolvement':         ['Low', 'Medium', 'High', 'Very High'],
    'JobSatisfaction':        ['Low', 'Medium', 'High', 'Very High'],
    'PerformanceRating':      ['Low', 'Good', 'Excellent', 'Outstanding'],
    'RelationshipSatisfaction':['Low', 'Medium', 'High', 'Very High'],
    'WorkLifeBalance':        ['Bad', 'Good', 'Better', 'Best'],
}

# Map integers → ordered Categorical strings
for col, mapping in ordinal_maps.items():
    df[col] = pd.Categorical(
        df[col].astype(int).map(mapping),
        categories=ordinal_orders[col],
        ordered=True
    )

# OHE the ordinal columns
df = pd.get_dummies(df, columns=list(ordinal_maps.keys()), drop_first=True)

# Convert all boolean columns to int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()


Shape: (1470, 69)
Missing values: 0


,Age,Attrition,DailyRate,DistanceFromHome,EmployeeNumber,HourlyRate,JobLevel,MonthlyIncome,MonthlyRate,NumCompaniesWorked,...,JobSatisfaction_Very High,PerformanceRating_Good,PerformanceRating_Excellent,PerformanceRating_Outstanding,RelationshipSatisfaction_Medium,RelationshipSatisfaction_High,RelationshipSatisfaction_Very High,WorkLifeBalance_Good,WorkLifeBalance_Better,WorkLifeBalance_Best
0,41,1,1102,1,1,94,2,5993,19479,8,...,1,0,1,0,0,0,0,0,0,0
1,49,0,279,8,2,61,2,5130,24907,1,...,0,0,0,1,0,0,1,0,1,0
2,37,1,1373,2,4,92,1,2090,2396,6,...,0,0,1,0,1,0,0,0,1,0
3,33,0,1392,3,5,56,1,2909,23159,1,...,0,0,1,0,0,1,0,0,1,0
4,27,0,591,2,7,40,1,3468,16632,9,...,0,0,1,0,0,0,1,0,1,0


In [10]:
import pickle

with open('df_cleaned.pkl', 'wb') as f:
    pickle.dump({'df': df}, f)
print(f"Saved df_cleaned.pkl — shape: {df.shape}")

Saved df_cleaned.pkl — shape: (1470, 69)
